<a href="https://colab.research.google.com/github/thegurdian/ML/blob/main/nlp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# ============================================================
# NLP SENTIMENT ANALYSIS
# Dataset: IMDb Movie Reviews
# Features: TF-IDF
# Model: Logistic Regression
# ============================================================

import re
import numpy as np
import pandas as pd

from tensorflow.keras.datasets import imdb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# STEP 1: LOAD DATASET
# ------------------------------------------------------------

# Keep the 10,000 most frequent words in the IMDb vocabulary
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(
    num_words=10000
)

# Convert integer-encoded reviews back into words
word_index = imdb.get_word_index()

index_to_word = {
    index + 3: word
    for word, index in word_index.items()
}

index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"


def decode_review(encoded_review):
    return " ".join(
        index_to_word.get(index, "<UNK>")
        for index in encoded_review
        if index != 0
    )


X_train_text = [decode_review(review) for review in X_train_raw]
X_test_text = [decode_review(review) for review in X_test_raw]

y_train = np.asarray(y_train)
y_test = np.asarray(y_test)

print("Training reviews:", len(X_train_text))
print("Testing reviews:", len(X_test_text))
print("Sample review:", X_train_text[0][:300])


# ------------------------------------------------------------
# STEP 2: TEXT PREPROCESSING
# ------------------------------------------------------------

def clean_text(text):
    text = text.lower()

    # Keep letters and spaces; remove punctuation and digits
    text = re.sub(r"[^a-z\s]", " ", text)

    # Normalize extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


X_train_clean = [clean_text(text) for text in X_train_text]
X_test_clean = [clean_text(text) for text in X_test_text]

print("\nOriginal review:")
print(X_train_text[0][:200])

print("\nCleaned review:")
print(X_train_clean[0][:200])


# ------------------------------------------------------------
# STEP 3: TF-IDF VECTORIZATION
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_df=0.95
)

# Fit ONLY on training data to prevent data leakage
X_train_tfidf = vectorizer.fit_transform(X_train_clean)

# Transform test data using the already-fitted vectorizer
X_test_tfidf = vectorizer.transform(X_test_clean)

print("\nTraining TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


# ------------------------------------------------------------
# STEP 4: TRAIN MODEL
# ------------------------------------------------------------

model = LogisticRegression(
    max_iter=1000,
    C=2.0,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("\nModel training completed!")


# ------------------------------------------------------------
# STEP 5: EVALUATE MODEL
# ------------------------------------------------------------

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# ------------------------------------------------------------
# STEP 6: PREDICT NEW REVIEWS
# ------------------------------------------------------------

def predict_sentiment(review):
    cleaned_review = clean_text(review)

    review_tfidf = vectorizer.transform([cleaned_review])

    prediction = model.predict(review_tfidf)[0]
    probabilities = model.predict_proba(review_tfidf)[0]

    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probabilities[prediction] * 100

    return sentiment, confidence


# Test with your own sentences
reviews = [
    "This movie was absolutely amazing and wonderful",
    "This movie was terrible and boring",
    "I really enjoyed the story and acting",
    "The film was a complete waste of time",
    "The movie was not good"
]

print("\nCUSTOM PREDICTIONS")

for review in reviews:
    sentiment, confidence = predict_sentiment(review)

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print("Model confidence:", round(confidence, 2), "%")


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training reviews: 25000
Testing reviews: 25000
Sample review: <START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the

Original review:
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and 

Cleaned review:
start this film was just brilliant casting location scenery story direction everyone s really suited the part they played and you could just imagine being there robert unk is an amazing actor and now 

Training TF-IDF shape: (25000, 20000)
Testing TF-IDF shape: (25000, 20000)
Vocabulary size: 20000

